# 02 - Variational Autoencoder Testing

Convolutional VAE for learning latent representations of CWT scalograms.
The trained encoder maps each scalogram to a low-dimensional vector;
downstream clustering (k-means) on those vectors identifies market regimes.

Pipeline: scalogram tensor -> ConvVAE encoder -> latent vectors -> k-means -> regime labels

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

## 1. Configuration

In [ ]:
# Paths (relative to notebooks/)
SCALOGRAM_PATH = Path("../data/processed/scalograms.npy")
METADATA_PATH  = Path("../data/processed/scalogram_metadata.csv")
FIGURE_DIR     = Path("../figures/vae/MNIST")
MODEL_DIR      = Path("../models/vae/MNIST")

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
Z_DIM      = 16
BETA       = 1.0
LR         = 1e-3
BATCH_SIZE = 64
EPOCHS     = 30
VAL_SPLIT  = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

NameError: name 'Path' is not defined

## 2. Data

The VAE receives image tensors with shape `(batch, channels, height, width)`,
values normalized to [0, 1].

`generate_scalograms.py` outputs a numpy array where each entry is one
scalogram (already min-max normalized to [0, 1]).
If the scalogram file is not available yet, the notebook falls back to
MNIST for validating the architecture.

In [ ]:
class ScalogramDataset(Dataset):
    def __init__(self, npy_path, metadata_path=None):
        data = np.load(npy_path)
        if data.ndim == 3:
            data = data[:, np.newaxis, :, :]  # (N, 1, H, W)
        self.images = torch.tensor(data, dtype=torch.float32)
        self.metadata = None
        if metadata_path and Path(metadata_path).exists():
            self.metadata = pd.read_csv(metadata_path)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], idx

In [ ]:
USE_MNIST = not SCALOGRAM_PATH.exists()

if USE_MNIST:
    from torchvision import datasets, transforms
    print("Scalogram file not found. Using MNIST as fallback.")
    transform = transforms.Compose([transforms.ToTensor()])
    full_dataset = datasets.MNIST(root="../data/cache", train=True, download=True, transform=transform)
    IMG_C, IMG_H, IMG_W = 1, 28, 28
else:
    full_dataset = ScalogramDataset(SCALOGRAM_PATH, METADATA_PATH)
    sample = full_dataset[0][0]
    IMG_C, IMG_H, IMG_W = sample.shape
    print(f"Loaded {len(full_dataset)} scalograms, shape: ({IMG_C}, {IMG_H}, {IMG_W})")

n_val = int(len(full_dataset) * VAL_SPLIT)
n_train = len(full_dataset) - n_val
train_set, val_set = random_split(full_dataset, [n_train, n_val])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {n_train}, Val: {n_val}")

In [ ]:
sample_batch, _ = next(iter(train_loader))
print(f"Batch shape: {sample_batch.shape}")
print(f"Value range: [{sample_batch.min():.3f}, {sample_batch.max():.3f}]")

fig, axes = plt.subplots(1, 6, figsize=(14, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(sample_batch[i, 0], cmap="viridis", aspect="auto")
    ax.axis("off")
fig.suptitle("Sample inputs", fontsize=12)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sample_inputs.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Architecture

```
Input (C, H, W)
      |
      v
  ENCODER
  Conv2d -> BN -> ReLU   x N layers (stride=2, halves spatial dims)
  Flatten
  Linear -> mu       (Z_DIM)
  Linear -> log_var  (Z_DIM)
      |
      v   Reparameterization: z = mu + sigma * epsilon,  epsilon ~ N(0,1)
      |
  DECODER
  Linear -> Unflatten
  ConvTranspose2d -> BN -> ReLU   x N layers
  Sigmoid
      |
      v
Output (C, H, W)
```

The encoder produces a distribution N(mu, sigma^2), not a fixed point.
The reparameterization trick makes sampling differentiable for backpropagation.

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, z_dim, in_channels, img_h, img_w):
        super().__init__()
        self.z_dim = z_dim
        self.img_h = img_h
        self.img_w = img_w

        # Build encoder layers dynamically based on input size
        # Each conv halves spatial dimensions (stride=2)
        filters = [32, 64, 128]
        encoder_layers = []
        ch_in = in_channels
        for ch_out in filters:
            encoder_layers += [
                nn.Conv2d(ch_in, ch_out, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(ch_out),
                nn.ReLU(),
            ]
            ch_in = ch_out
        encoder_layers.append(nn.Flatten())
        self.encoder = nn.Sequential(*encoder_layers)

        # Compute flattened size after encoder convolutions
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, img_h, img_w)
            self.feat_shape = self.encoder[:-1](dummy).shape[1:]  # (C, H', W')
            flat_dim = int(np.prod(self.feat_shape))

        self.fc_mu      = nn.Linear(flat_dim, z_dim)
        self.fc_log_var = nn.Linear(flat_dim, z_dim)

        # Decoder
        self.decoder_proj = nn.Linear(z_dim, flat_dim)

        decoder_layers = []
        rev_filters = list(reversed(filters))
        for i, ch_out in enumerate(rev_filters[1:] + [in_channels]):
            ch_in = rev_filters[i]
            if i < len(rev_filters) - 1:
                decoder_layers += [
                    nn.ConvTranspose2d(ch_in, ch_out, kernel_size=3, stride=2, padding=1, output_padding=1),
                    nn.BatchNorm2d(ch_out),
                    nn.ReLU(),
                ]
            else:
                decoder_layers += [
                    nn.ConvTranspose2d(ch_in, ch_out, kernel_size=3, stride=2, padding=1, output_padding=1),
                    nn.Sigmoid(),
                ]
        self.decoder = nn.Sequential(*decoder_layers)

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_log_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        h = self.decoder_proj(z)
        h = h.view(-1, *self.feat_shape)
        return self.decoder(h)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        x_recon = self.decode(z)
        x_recon = x_recon[:, :, :x.shape[2], :x.shape[3]]
        return x_recon, mu, log_var

In [ ]:
model = ConvVAE(z_dim=Z_DIM, in_channels=IMG_C, img_h=IMG_H, img_w=IMG_W).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total_params:,}")
print(f"Latent dim: {Z_DIM}")
print(f"Encoder feature shape: {model.feat_shape}")
print()
print(model)

## 4. Loss function

**Loss = Reconstruction + beta * KL**

The reconstruction term (BCE) measures how well the decoder reproduces the input.
The KL term measures how far the learned distribution q(z|x) = N(mu, sigma^2)
is from the prior p(z) = N(0, 1).

beta controls the balance:
- beta << 1: sharp reconstructions, unstructured latent space (bad for clustering)
- beta = 1: standard VAE
- beta > 1: blurry reconstructions, well-organized latent space (beta-VAE)

In [ ]:
def vae_loss(x_recon, x, mu, log_var, beta=1.0):
    recon = F.binary_cross_entropy(x_recon, x, reduction="sum") / x.shape[0]
    kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.shape[0]
    return recon + beta * kl, recon, kl

## 5. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train_loss": [], "train_recon": [], "train_kl": [],
           "val_loss": [],   "val_recon": [],   "val_kl": []}

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    t_loss, t_recon, t_kl = 0, 0, 0
    for batch_x, _ in train_loader:
        batch_x = batch_x.to(DEVICE)
        x_recon, mu, log_var = model(batch_x)
        loss, recon, kl = vae_loss(x_recon, batch_x, mu, log_var, beta=BETA)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        t_loss  += loss.item()
        t_recon += recon.item()
        t_kl    += kl.item()

    # Validate
    model.eval()
    v_loss, v_recon, v_kl = 0, 0, 0
    with torch.no_grad():
        for batch_x, _ in val_loader:
            batch_x = batch_x.to(DEVICE)
            x_recon, mu, log_var = model(batch_x)
            loss, recon, kl = vae_loss(x_recon, batch_x, mu, log_var, beta=BETA)
            v_loss  += loss.item()
            v_recon += recon.item()
            v_kl    += kl.item()

    for key, val, n in [
        ("train_loss", t_loss, len(train_loader)), ("train_recon", t_recon, len(train_loader)),
        ("train_kl", t_kl, len(train_loader)),     ("val_loss", v_loss, len(val_loader)),
        ("val_recon", v_recon, len(val_loader)),    ("val_kl", v_kl, len(val_loader)),
    ]:
        history[key].append(val / n)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS}  "
              f"Train: {history['train_loss'][-1]:.1f} (R:{history['train_recon'][-1]:.1f} KL:{history['train_kl'][-1]:.1f})  "
              f"Val: {history['val_loss'][-1]:.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = [("Total loss", "train_loss", "val_loss"),
          ("Reconstruction", "train_recon", "val_recon"),
          ("KL divergence", "train_kl", "val_kl")]
for ax, (title, tk, vk) in zip(axes, labels):
    ax.plot(history[tk], label="train")
    ax.plot(history[vk], label="val", linestyle="--")
    ax.set_xlabel("Epoch")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "z_dim": Z_DIM,
    "beta": BETA,
    "img_shape": (IMG_C, IMG_H, IMG_W),
    "epochs": EPOCHS,
    "history": history,
}, MODEL_DIR / "vae.pt")
print(f"Model saved to {MODEL_DIR / 'vae.pt'}")

## 6. Reconstruction quality

If reconstructions are blurry but recognizable, the VAE works correctly.
Identical reproductions suggest the KL term is too weak (overfitting).
Pure noise means the model did not learn.

In [ ]:
model.eval()
with torch.no_grad():
    sample_x, _ = next(iter(val_loader))
    sample_x = sample_x.to(DEVICE)
    recon_x, _, _ = model(sample_x)

n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(16, 4))
for i in range(n_show):
    axes[0, i].imshow(sample_x[i, 0].cpu(), cmap="viridis", aspect="auto")
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon_x[i, 0].cpu(), cmap="viridis", aspect="auto")
    axes[1, i].set_title("Reconstructed")
    axes[1, i].axis("off")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Latent space extraction

After training, each scalogram is passed through the encoder.
The mu vector (mean of the learned distribution) is used as the
fixed-point representation for clustering.

In [ ]:
def extract_latents(model, dataloader, device):
    model.eval()
    all_mu, all_idx = [], []
    with torch.no_grad():
        for x, idx in dataloader:
            x = x.to(device)
            mu, _ = model.encode(x)
            all_mu.append(mu.cpu().numpy())
            all_idx.append(idx.numpy() if isinstance(idx, torch.Tensor) else np.array(idx))
    return np.concatenate(all_mu), np.concatenate(all_idx)

full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=False)
latents, indices = extract_latents(model, full_loader, DEVICE)
print(f"Latent vectors: {latents.shape}")

## 8. Clustering

K-means on the latent vectors. Silhouette score helps choose k:
higher is better, typical range [0.2, 0.7] for real data.

In [ ]:
k_range = range(2, 9)
silhouettes = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(latents)
    s = silhouette_score(latents, labels)
    silhouettes.append(s)
    print(f"k={k}  silhouette={s:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(k_range), silhouettes, "o-")
ax.set_xlabel("k")
ax.set_ylabel("Silhouette score")
ax.set_title("Silhouette score vs number of clusters")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "silhouette_scores.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
K_BEST = int(k_range[np.argmax(silhouettes)])
print(f"Best k by silhouette: {K_BEST}")

kmeans = KMeans(n_clusters=K_BEST, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(latents)

print(f"Cluster sizes: {np.bincount(cluster_labels)}")

## 9. Latent space visualization

In [ ]:
n_plot = min(5000, len(latents))
rng = np.random.default_rng(42)
plot_idx = rng.choice(len(latents), size=n_plot, replace=False)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
latents_2d = tsne.fit_transform(latents[plot_idx])

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(latents_2d[:, 0], latents_2d[:, 1],
                     c=cluster_labels[plot_idx], cmap="tab10", s=8, alpha=0.6)
ax.set_title(f"t-SNE of latent space (k={K_BEST})")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
plt.colorbar(scatter, ax=ax, label="Cluster")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "tsne_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Regime timeline

Map cluster labels back to the original time axis using the metadata CSV.
This section only runs when using actual scalogram data (not MNIST).

In [ ]:
if not USE_MNIST and METADATA_PATH.exists():
    meta = pd.read_csv(METADATA_PATH)
    meta["cluster"] = cluster_labels

    fig, ax = plt.subplots(figsize=(16, 3))
    ax.scatter(pd.to_datetime(meta["start_time"]), meta["cluster"],
               c=meta["cluster"], cmap="tab10", s=4, alpha=0.7)
    ax.set_xlabel("Date")
    ax.set_ylabel("Regime")
    ax.set_title("Detected regimes over time")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "regime_timeline.png", dpi=150, bbox_inches="tight")
    plt.show()

    meta.to_csv("../data/processed/regime_labels.csv", index=False)
    print(f"Regime labels saved to data/processed/regime_labels.csv")
else:
    print("Skipped: requires scalogram metadata.")

## 11. Save latent vectors

In [ ]:
np.save(MODEL_DIR / "latent_vectors.npy", latents)
np.save(MODEL_DIR / "cluster_labels.npy", cluster_labels)
print(f"Latent vectors: {MODEL_DIR / 'latent_vectors.npy'}")
print(f"Cluster labels: {MODEL_DIR / 'cluster_labels.npy'}")